In [1]:
import pandas as pd
import os
import sqlite3
from sqlite3 import Error

In [2]:
from constants import * 
from utils import *

In [3]:
print(DB_PATH)
print(DB_FILE_NAME)

/home/airflow/dags/Lead_scoring_data_pipeline/
lead_scoring_data_cleaning.db


In [4]:
con = sqlite3.connect(DB_PATH + DB_FILE_NAME)
df = pd.read_sql(f"SELECT name FROM sqlite_master WHERE type='table'", con)
print(df.shape)
df.head()

(7, 1)


,name
0,loaded_data
1,city_tier_mapped
2,categorical_variables_mapped
3,interactions_mapped
4,model_input


In [5]:
df = pd.read_sql('select * from interactions_mapped', con)
print(df.shape)
df.head()

(238964, 13)


,created_date,city_tier,first_platform_c,first_utm_medium_c,first_utm_source_c,total_leads_droppped,referred_lead,app_complete_flag,assistance_interaction,career_interaction,payment_interaction,social_interaction,syllabus_interaction
0,2021-07-01 00:08:15,1.0,Level0,Level11,Level2,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
1,2021-07-01 00:16:43,2.0,Level3,Level0,others,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
2,2021-07-01 00:22:20,1.0,Level3,Level0,Level0,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
3,2021-07-01 00:23:13,1.0,Level1,Level3,others,2.0,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2021-07-01 00:28:38,1.0,Level3,Level0,Level0,1.0,0.0,0,0.0,0.0,0.0,0.0,0.0


In [6]:
encoded_df = df[FEATURES_TO_ENCODE]
encoded_df.head()

,city_tier,first_platform_c,first_utm_medium_c,first_utm_source_c
0,1.0,Level0,Level11,Level2
1,2.0,Level3,Level0,others
2,1.0,Level3,Level0,Level0
3,1.0,Level1,Level3,others
4,1.0,Level3,Level0,Level0


In [7]:
columns = df.columns.to_list()
print(columns)
print(len(columns))

['created_date', 'city_tier', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c', 'total_leads_droppped', 'referred_lead', 'app_complete_flag', 'assistance_interaction', 'career_interaction', 'payment_interaction', 'social_interaction', 'syllabus_interaction']
13


In [8]:
%time
df_new_encoded = pd.get_dummies(df, columns = FEATURES_TO_ENCODE,drop_first=True)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 10.5 µs


In [9]:
columns = df_new_encoded.columns.to_list()
print(columns)
print(len(columns))

['created_date', 'total_leads_droppped', 'referred_lead', 'app_complete_flag', 'assistance_interaction', 'career_interaction', 'payment_interaction', 'social_interaction', 'syllabus_interaction', 'city_tier_2.0', 'city_tier_3.0', 'first_platform_c_Level1', 'first_platform_c_Level2', 'first_platform_c_Level3', 'first_platform_c_Level7', 'first_platform_c_Level8', 'first_platform_c_others', 'first_utm_medium_c_Level10', 'first_utm_medium_c_Level11', 'first_utm_medium_c_Level13', 'first_utm_medium_c_Level15', 'first_utm_medium_c_Level16', 'first_utm_medium_c_Level2', 'first_utm_medium_c_Level20', 'first_utm_medium_c_Level26', 'first_utm_medium_c_Level3', 'first_utm_medium_c_Level30', 'first_utm_medium_c_Level33', 'first_utm_medium_c_Level4', 'first_utm_medium_c_Level43', 'first_utm_medium_c_Level5', 'first_utm_medium_c_Level6', 'first_utm_medium_c_Level8', 'first_utm_medium_c_Level9', 'first_utm_medium_c_others', 'first_utm_source_c_Level14', 'first_utm_source_c_Level16', 'first_utm_sourc

In [10]:
FEATURES_TO_ENCODE

['city_tier', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c']

In [11]:
encode_features()

## Verify data written to table

In [12]:
df = pd.read_sql(f"SELECT name FROM sqlite_master WHERE type='table'", con)
print(df.shape)
df.head()

(7, 1)


,name
0,loaded_data
1,city_tier_mapped
2,categorical_variables_mapped
3,interactions_mapped
4,model_input


In [13]:
con = sqlite3.connect(DB_PATH + DB_FILE_NAME)
X = pd.read_sql('select * from features', con)
X.head()

,total_leads_droppped,referred_lead,city_tier_1.0,city_tier_2.0,city_tier_3.0,first_platform_c_Level0,first_platform_c_Level3,first_platform_c_Level7,first_platform_c_Level1,first_platform_c_Level2,...,first_utm_medium_c_others,first_utm_source_c_Level2,first_utm_source_c_Level0,first_utm_source_c_Level7,first_utm_source_c_Level4,first_utm_source_c_Level6,first_utm_source_c_Level16,first_utm_source_c_Level5,first_utm_source_c_Level14,first_utm_source_c_others
0,1.0,0.0,1,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,1.0,0.0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,1.0,0.0,1,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,2.0,0.0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,1.0,0.0,1,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [14]:
y = pd.read_sql('select * from target', con)
y.head()

,app_complete_flag
0,1
1,1
2,1
3,0
4,0


## create sql lite connection and start database


In [15]:
# create a connection and setup a SQLite database with the name "lead_scoring_model_experimentation.db" in 
# 'Assignment/02_training_pipeline/notebooks/' location
import sqlite3
from sqlite3 import Error

def create_connection(file):
    #create a database connection to a SQLite database
    connection = None
    try:
        connection = sqlite3.connect(file)
        print(sqlite3.version)
    except Error as e:
        print(e)
    finally:
        if connection:
            connection.close()

if __name__ == '__main__':
    create_connection("Lead_scoring_mlflow_production.db")

2.6.0


In [16]:
##mlflow server --backend-store-uri='sqlite:////home/Assignment/02_training_pipeline/scripts/Lead_scoring_mlflow_production.db' --default-artifact-root="file:///home/Assignment/02_training_pipeline/scripts/mlruns/" --port=6006 --host=0.0.0.0

In [17]:
mlflow.set_tracking_uri("http://0.0.0.0:6006")


In [18]:
get_trained_model()

2025/02/14 16:30:58 INFO mlflow.tracking.fluent: Experiment with name 'Lead_scoring_mlflow_production' does not exist. Creating a new experiment.
/opt/conda/lib/python3.8/site-packages/sklearn/utils/validation.py:72: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
Registered model 'LightGBM' already exists. Creating a new version of this model...
2025/02/14 16:31:13 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: LightGBM, version 4
Created version '4' of model 'LightGBM'.


get_trained_model has been executed successfully.
